# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** SaaS Product Analytics, Cohorts & Customer Lifetime Value (LTV)

---
*Note: this notebook operationalizes foundational unit-economics metrics for an IoT/SaaS business with interconnected hardware and subscription lines. It runs on a synthetic data warehouse generated by `src/data_generator.py`; the figures below are illustrative of the method, not measurements of a real company.*

*The methodological spine of the notebook is one distinction that is easy to get wrong, and expensive to get wrong: the fraction of subscribers who have cancelled to date is a cumulative share, and a cumulative share is the wrong number for the LTV formula, since it grows the longer a cohort has been watched and has nothing to do with the pace of loss. The **per-period hazard** is the right number, because $1/\text{hazard}$ is only a lifetime if hazard is a rate, not a running tally. I compute the tempting-but-wrong number first, then derive the dimensionally correct monthly hazard from subscriber-months of exposure (right-censoring handled properly), and show the two produce LTV estimates that differ by roughly an order of magnitude. The generator plants a constant monthly churn hazard (Camera 0.040, Doorbell 0.075), so the corrected estimate is checkable against ground truth.*

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("crest")

### 1. Simulated Data Warehouse Ingestion (Hardware & Subscriptions)
I load the transactional database of device sales (Security Cameras vs Smart Doorbells) and cloud storage contractual records (Premium Subscriptions).

In [2]:
# Dimension Table: Users and Installed Base Inventory
df_users = pd.read_csv('../data/hardware_users.csv')
df_users['PurchaseDate'] = pd.to_datetime(df_users['PurchaseDate'])

# Fact Table: Subscription Records
df_subs = pd.read_csv('../data/subscriptions.csv')
df_subs['SubscriptionStart'] = pd.to_datetime(df_subs['SubscriptionStart'])
df_subs['SubscriptionEnd'] = pd.to_datetime(df_subs['SubscriptionEnd'])

# Hybrid Join: Enrich installed base with subscription status
# SQL Equivalent: SELECT * FROM Users LEFT JOIN Subs ON Users.UserID = Subs.UserID
df_analytics = pd.merge(df_users, df_subs, on='UserID', how='left')

print(f"[*] Total Installed Base: {len(df_users)} Active Hardware.")
print(f"[*] Subscriber Base: {len(df_subs)} Processed contracts.")
df_analytics.head()

[*] Total Installed Base: 5000 Active Hardware.
[*] Subscriber Base: 1403 Processed contracts.


,UserID,DeviceType,PurchaseDate,OnboardingVersion,CohortMonth,PlanName,MonthlyFee,SubscriptionStart,SubscriptionEnd,IsActive
0,10000,Security Camera,2022-02-20,Control,2022-02,Premium Cloud Storage,14.99,2022-02-22,2023-03-06,False
1,10001,Smart Doorbell,2022-03-12,Variant,2022-03,NaN,NaN,NaT,NaT,NaN
2,10002,Smart Doorbell,2022-02-17,Variant,2022-02,Premium Cloud Storage,9.99,2022-02-22,2022-06-24,False
3,10003,Smart Doorbell,2022-01-19,Variant,2022-01,NaN,NaN,NaT,NaT,NaN
4,10004,Security Camera,2022-09-19,Variant,2022-09,NaN,NaN,NaT,NaT,NaN


### 2. Conversions and Hardware-to-Subscription Attach Rate
The *Attach Rate* measures the effectiveness of the *in-app* funnel in converting a recent physical buyer into a recurring value user.

In [3]:
total_hardware = len(df_users)
total_subscriptions = len(df_subs)
global_attach_rate = (total_subscriptions / total_hardware) * 100

print(f"[KPI] Global Hardware-to-Subscription Attach Rate: {global_attach_rate:.2f}%")

conv_by_device = df_analytics.groupby('DeviceType').agg(
    Total_Hardware=('UserID', 'count'),
    Subscribers=('SubscriptionStart', 'count')
)
conv_by_device['Attach_Rate_Pct'] = (conv_by_device['Subscribers'] / conv_by_device['Total_Hardware']) * 100
conv_by_device


[KPI] Global Hardware-to-Subscription Attach Rate: 28.06%


,Total_Hardware,Subscribers,Attach_Rate_Pct
DeviceType,,,
Security Camera,2033,762,37.481554
Smart Doorbell,2967,641,21.604314


### 3. Cancellations to date — a cumulative share, *not* a churn rate
The first summary one reaches for is the share of each segment's subscribers that have cancelled by the data-extraction date. It is fine for a dashboard, but it is **not** the monthly churn rate the LTV formula needs. It is a cumulative proportion: it grows the longer a cohort has been observed and is pulled *down* by right-censoring (subscribers still active have simply not had the chance to churn yet). I compute it here to set up the contrast, then derive the correct per-period rate in the next section.

In [4]:
# Subscribers only (rows with an actual SubscriptionStart)
df_subscribers = df_analytics[df_analytics['SubscriptionStart'].notna()].copy()

# Cumulative cancellation share = cancelled-to-date / signups. This is the number
# the naive LTV reaches for -- and it is the WRONG one: a window-length- and
# censoring-dependent PROPORTION, not a monthly RATE.
cancel_share = df_subscribers.groupby('DeviceType').agg(
    Signups=('UserID', 'count'),
    Cancelled_To_Date=('IsActive', lambda x: (x == False).sum())
)
cancel_share['Cancelled_Share_Pct'] = (cancel_share['Cancelled_To_Date'] / cancel_share['Signups']) * 100

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=cancel_share.reset_index(), x='DeviceType', y='Cancelled_Share_Pct',
            palette=['#ff5252', '#4db6ac'], ax=ax)
ax.set_title("Cumulative cancellation share to date — NOT a monthly churn rate", fontsize=13, pad=10)
ax.set_ylabel("Subscribers cancelled to date (%)")
ax.set_xlabel("")
for i, v in enumerate(cancel_share['Cancelled_Share_Pct']):
    ax.text(i, v - 3, f"{v:.1f}%", ha='center', color='white', fontweight='bold', fontsize=12)
plt.show()


### 4. The dimensional fix: a monthly hazard from subscriber-months of exposure
The LTV identity $LTV = \dfrac{ARPU \cdot m}{\text{churn}}$ holds only when *churn* is a per-period rate, a hazard: then $1/\text{churn}$ is the expected lifetime measured *in those periods*. Feed the formula the cumulative cancellation share instead and it silently treats a dimensionless proportion as a monthly rate, so $1/\text{share}$ has no coherent meaning — there is no unit in which that number is a duration.

The right tool is the **occurrence–exposure hazard**: total churn events divided by total subscriber-months of exposure. It handles right-censoring for free, since a still-active subscriber adds exposure to the denominator without ever adding an event to the numerator. Under a constant monthly hazard $h$ the lifetime is geometric, so the mean lifetime is $1/h$ months and the survival curve is $S(t)=(1-h)^t$.

In [5]:
# Warehouse extraction date: subscriptions still active are right-censored here.
SNAPSHOT = pd.Timestamp('2023-05-31')

surv = df_subscribers.copy()
surv['Churned'] = (surv['IsActive'] == False)
# Exposure in months: start -> churn (observed) or start -> snapshot (censored).
end_obs = surv['SubscriptionEnd'].fillna(SNAPSHOT)
surv['Exposure_Months'] = ((end_obs - surv['SubscriptionStart']).dt.days / 30.44).clip(lower=0.01)

# Occurrence-exposure hazard: churn events per subscriber-month of exposure.
# This is the dimensionally-correct per-period rate, and it absorbs censoring:
# active subscribers contribute exposure (denominator) but no event (numerator).
hazard = surv.groupby('DeviceType').agg(
    Churn_Events=('Churned', 'sum'),
    Exposure_Months=('Exposure_Months', 'sum')
)
hazard['Monthly_Hazard'] = hazard['Churn_Events'] / hazard['Exposure_Months']
hazard['Mean_Lifetime_Months'] = 1.0 / hazard['Monthly_Hazard']

print("[*] Occurrence-exposure monthly churn hazard (the rate the LTV formula needs):")
print(hazard.round({'Monthly_Hazard': 4, 'Mean_Lifetime_Months': 1}))
print("\n[i] Synthetic ground truth planted in the generator: Camera h=0.040, "
      "Doorbell h=0.075 (Variant onboarding x0.90).")

# Fitted geometric retention curve S(t) = (1 - h)^t implied by each hazard.
fig, ax = plt.subplots(figsize=(9, 5))
t = np.arange(0, 37)
for dev, colour in [('Security Camera', '#4db6ac'), ('Smart Doorbell', '#ff7043')]:
    h = hazard.loc[dev, 'Monthly_Hazard']
    ax.plot(t, (1 - h) ** t, color=colour, lw=2.4,
            label=f"{dev}  —  h={h:.3f}/mo, mean life {1/h:.0f} mo")
ax.axhline(1 / np.e, ls=':', color='grey', lw=1)
ax.text(36, 1 / np.e + 0.01, '1/e', color='grey', ha='right', fontsize=9)
ax.set_title("Fitted geometric retention curve by hardware line", fontsize=13, pad=10)
ax.set_xlabel("Months since subscription start")
ax.set_ylabel(r"Probability still subscribed  $S(t)=(1-h)^t$")
ax.set_ylim(0, 1.02)
ax.legend(frameon=False)
plt.show()


[*] Occurrence-exposure monthly churn hazard (the rate the LTV formula needs):
                 Churn_Events  Exposure_Months  Monthly_Hazard  \
DeviceType                                                       
Security Camera           224      6736.169514          0.0333   
Smart Doorbell            325      4659.001314          0.0698   

                 Mean_Lifetime_Months  
DeviceType                             
Security Camera                  30.1  
Smart Doorbell                   14.3  

[i] Synthetic ground truth planted in the generator: Camera h=0.040, Doorbell h=0.075 (Variant onboarding x0.90).


### 5. LTV done correctly — and the size of the error
With the monthly hazard $\hat h$ in hand the lifetime-value identity is now dimensionally sound:

$$ LTV = \frac{ARPU \times \text{margin}}{\hat h}, \qquad \mathbb{E}[\text{lifetime}] = \frac{1}{\hat h}\ \text{months}. $$

I assume an 80% gross margin on cloud storage. Alongside the correct figure I recompute the **naive** one (dividing by the cumulative cancellation share, the original mistake) to quantify how far it lands from the truth.

In [6]:
GROSS_MARGIN = 0.80  # gross margin on software / cloud storage

arpu = df_subscribers.groupby('DeviceType')['MonthlyFee'].mean().rename('ARPU')

ltv = pd.concat([
    arpu,
    hazard[['Monthly_Hazard', 'Mean_Lifetime_Months']],
    (cancel_share['Cancelled_Share_Pct'] / 100).rename('Cumulative_Cancel_Share'),
], axis=1)

# Correct: divide by the per-period hazard (a rate -> 1/h is a lifetime in months).
ltv['LTV_Correct_USD'] = (ltv['ARPU'] * GROSS_MARGIN) / ltv['Monthly_Hazard']
# Naive (the dimensional error): divide by the cumulative cancellation share.
ltv['LTV_Naive_USD'] = (ltv['ARPU'] * GROSS_MARGIN) / ltv['Cumulative_Cancel_Share']
ltv['Overstatement_x'] = ltv['LTV_Correct_USD'] / ltv['LTV_Naive_USD']

print(ltv[['ARPU', 'Monthly_Hazard', 'Mean_Lifetime_Months',
           'LTV_Correct_USD', 'LTV_Naive_USD', 'Overstatement_x']].round(2))

ratio = ltv.loc['Security Camera', 'LTV_Correct_USD'] / ltv.loc['Smart Doorbell', 'LTV_Correct_USD']
print(f"\n[KPI] Corrected LTV: Camera ${ltv.loc['Security Camera','LTV_Correct_USD']:.0f} "
      f"vs Doorbell ${ltv.loc['Smart Doorbell','LTV_Correct_USD']:.0f}  ->  {ratio:.1f}x.")
print("[!] The naive formula understates LTV by roughly an order of magnitude because it "
      "divides by a cumulative proportion instead of a monthly rate.")


                  ARPU  Monthly_Hazard  Mean_Lifetime_Months  LTV_Correct_USD  \
DeviceType                                                                      
Security Camera  14.99            0.03                 30.07           360.63   
Smart Doorbell    9.99            0.07                 14.34           114.57   

                 LTV_Naive_USD  Overstatement_x  
DeviceType                                       
Security Camera          40.79             8.84  
Smart Doorbell           15.76             7.27  

[KPI] Corrected LTV: Camera $361 vs Doorbell $115  ->  3.1x.
[!] The naive formula understates LTV by roughly an order of magnitude because it divides by a cumulative proportion instead of a monthly rate.


### 6. A heatmap of conversion, not of churn
Cohort analysis here asks a narrower question than the one above: did product, marketing, or onboarding changes shift the **attach rate** (the conversion from hardware buyer to subscriber) across acquisition months? This heatmap is not a retention curve and was never meant to be one: it tracks who signed up, not how long they stayed, so a cold month here says something about the funnel, not about the churn hazard estimated in section 4. The two lenses are complementary precisely because they answer different questions with different clocks.

In [7]:
cohort_data = df_analytics.groupby(['CohortMonth', 'DeviceType']).agg(
    Total_Hardware=('UserID', 'count'),
    Subscribers=('SubscriptionStart', 'count')
).reset_index()

cohort_data['Attach_Rate'] = (cohort_data['Subscribers'] / cohort_data['Total_Hardware'])

cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='DeviceType', values='Attach_Rate')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cohort_pivot, annot=True, fmt=".1%", cmap="YlGnBu_r", linewidths=.5, cbar_kws={'label': 'SaaS Conversion Rate'})
ax.set_title("Cohort Evolution of Subscription Attach Rate (2022)", fontsize=15, pad=15)
ax.set_ylabel("Acquisition Cohort (Purchase Month)")
ax.set_xlabel("Hardware Core")
plt.show()

### Synthesis — and why the dimensional care matters for the business
On this synthetic warehouse the qualitative story holds up: Security Cameras combine a higher monthly fee with a markedly lower churn hazard, which gives them a lifetime value of about \$361 against \$115 for Smart Doorbells (roughly 3×). The naive computation reported the same ordering, so it would be easy to shrug off the dimensional fuss as pedantry.

It isn't, and the reason is the absolute number, not the ranking. Dividing by the cumulative cancellation share understated LTV by 7–9×: \$41 naive versus \$361 correct for the camera, \$16 versus \$115 for the doorbell. A media team sizing acquisition spend against that naive \$41 camera figure would cap CAC at roughly a ninth of what the segment can actually support, and would do it silently — the error never shows up in the ranking, only in the budget, which is exactly where you don't want to find it. Recovered hazards land close to the planted ground truth (≈0.033 and ≈0.070 against 0.040 and 0.075 planted; the camera estimate sits a touch low because cameras churn rarely and spend most of their lives censored, so fewer events anchor the estimate). Those are the numbers an LTV:CAC ratio should be built on, not the naive ones.

**Reading limits.** Every figure here is synthetic. The LTV identity assumes a constant monthly hazard (a geometric lifetime), and I'd expect a real product to show duration dependence instead: early-life churn spikes, a loyal tail that survives long past what geometric decay predicts, the kind of shape a Kaplan–Meier curve or a parametric survival fit would catch and this model can't. Margin (80%) is assumed rather than measured, discounting is omitted, and the cohort heatmap upstream measures attach rate, not retention. None of that changes the discipline this notebook argues for: before you divide anything into an LTV formula, ask whether what you're holding is a per-period rate or a cumulative share dressed up to look like one.